In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-5-nano")

In [2]:
@tool
def save_book_info():
    """주어진 책 제목을 저장하는 도구"""
    # 실제로는 데이터베이스나 파일에 저장하는 로직이 들어감
    return f"책  정보가 저장되었습니다."


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model,
    tools=[save_book_info],
    middleware=[
        # 이메일은 완전히 가림 (redact)
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True),
        # 신용카드 번호는 마스킹 처리 (mask)
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True),
    ],
)


In [4]:
prompt = "안녕하세요. 이메일은 user123@example.com 입니다. 제 카드번호는 1234123443214321 입니다."
response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})

# 에이전트가 실제로 넘겨받은(마스킹된) 메시지 확인
print(response)
print(response["messages"][0].content)

{'messages': [HumanMessage(content='안녕하세요. 이메일은 [REDACTED_EMAIL] 입니다. 제 카드번호는 ************4321 입니다.', additional_kwargs={}, response_metadata={}, id='20afd44a-6d08-4805-8adf-55cc3e014d78'), AIMessage(content='안녕하세요!\n\n입력하신 이메일과 카드번호 관련해서 한 가지 안내드립니다. 이 채널에 신용카드 번호를 포함한 민감한 정보를 입력하거나 저장하는 것은 보안상 위험할 수 있습니다. 특히 카드번호 전체나 부분을 공유하는 일은 피하는 것이 좋습니다. 필요하신 경우 안전한 방법으로 처리할 수 있도록 도와드리겠습니다.\n\n원하시는 도움을 구체적으로 알려주시면 보안에 신경 써서 안내해 드리겠습니다. 예를 들어:\n- 이메일 주소나 계정 정보 수정/확인\n- 결제 정보 안전한 업데이트 방법 안내 (공식 결제 페이지 이용 권고)\n- 책 제목 저장 등 비민감한 정보 처리\n- 필요하다면 마지막 네 자리를 확인하는 용도의 일반적인 절차 안내\n\n또한 원하시면 책 제목 저장 기능 같은 비민감한 작업은 진행할 수 있습니다. 저장할 제목을 알려주시면 도와드리겠습니다.\n\n어떤 도움을 원하시는지 구체적으로 알려주실래요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1321, 'prompt_tokens': 147, 'total_tokens': 1468, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1088, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cac

In [ ]:
# 휴대폰 번호(010-XXXX-XXXX)를 잡아내는 커스텀 정규식 패턴
phone_number_regex = r"\b(010)[-\s]?(\d{3,4})[-\s]?(\d{4})\b"

phone_masking_middleware = PIIMiddleware(
    pii_type="phone_number",
    detector=phone_number_regex,
    strategy="mask",
    apply_to_input=True,
)


In [ ]:
import re
# **re**는 파이썬의 내장 라이브러리로, **정규 표현식(Regular Expression)**을 다루기 위한 도구입니다.
# re: 복잡한 문자열 패턴을 찾기 위한 파이썬 기본 도구 상자.

def detect_api_key(content: str) -> list[dict[str, str | int]]:
    """'sk-'로 시작하는 32자리 문자열을 API 키로 간주하고 탐지합니다."""
    matches = []
    pattern = r"sk-[a-zA-Z0-9]{32}"
    for match in re.finditer(pattern, content):
        matches.append({
            "text": match.group(0), # 정규 표현식 패턴과 일치하는 전체 텍스트를 가져옵니다.
            "start": match.start(), # 일치하는 패턴이 시작되는 문자열 내의 인덱스(위치) 번호
            "end": match.end(),     # 일치하는 패턴이 끝나는 바로 다음 위치의 인덱스 번호입니다.
        })
    return matches


# pattern = r"..." 에서 r의 의미
# 문자열 앞에 붙은 **r**은 **Raw String(원시 문자열)**을 의미합니다.

#왜 쓰나요? 파이썬 문자열에서 백슬래시(\)는 특별한 의미(예: \n은 줄바꿈)를 가집니다. 하지만 정규 표현식에서는 백슬래시를 규칙의 일부로 자주 사용합니다.
# r을 붙이면: 파이썬이 문자열 안의 백슬래시를 해석하지 않고 적힌 그대로(Raw) 정규 표현식 엔진에 전달하게 됩니다.

# 예시:
# 일반 문자열: "C:\Users\name" -> \U를 특수 문자로 오해해서 에러가 날 수 있음.
# Raw String: r"C:\Users\name" -> 적힌 그대로의 경로로 인식함.